# 启发式算法（遗传 / 模拟退火）：一个模型一个 Notebook

## 什么时候用
目标函数**非凸、不可导、或根本没解析式**时（比如"给一套排班打分""给一条路径打分"），
梯度类方法（SLSQP）会失效，这时用无梯度启发式：

| 算法 | 思路 | 特点 |
| --- | --- | --- |
| **遗传算法 GA** | 种群 → 选择 → 交叉 → 变异 → 迭代 | 全局搜索能力强，参数多（种群、代数、变异率） |
| **模拟退火 SA** | 随机扰动，按概率接受变差的解，温度逐渐降低 | 实现简单，适合组合优化 |

## 必须诚实的一点
启发式**不保证全局最优**。论文里必须报告：
- 运行了几次、每次结果差多少（结果分布）；
- 与精确解/下界的差距（能算的话）；
- 参数（种群、代数、温度）是怎么定的。

只报一次"最好结果"，评委会怀疑是挑出来的。

## 目录
0 环境 / 1 配置 / 2 目标函数与工具 / 3 GA / 4 SA / 5 对比与收敛 / 6 导出

> 只依赖 `numpy`，不需要额外装 `deap`、`scikit-opt` 之类的库。


In [1]:
# -*- coding: utf-8 -*-
# =====================================================================
# 0. 环境与路径（每个 Notebook 都带这一段，保证单独打开就能跑）
# =====================================================================
# 依赖：numpy、pandas、scipy（>=1.9，用到 milp）
# 可选：matplotlib（画图）、openpyxl（导出 xlsx）
# 缺哪个就装：pip install numpy pandas scipy matplotlib openpyxl
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import optimize as sp_opt
from scipy.optimize import Bounds, LinearConstraint

try:
    import matplotlib
    import matplotlib.pyplot as plt
    # 不设中文字体，图里的标题和标签会变成方块
    plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "DejaVu Sans"]
    plt.rcParams["axes.unicode_minus"] = False
    HAS_MPL = True
except ImportError:
    HAS_MPL = False


def find_root(marker="导入数据"):
    """向上逐级找含 导入数据/ 的目录。

    Notebook 放在 优化类模型/ 下也能找到项目根，换电脑、换目录都不用改死路径。
    """
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / marker).is_dir():
            return p
    return Path.cwd()


ROOT = find_root()
DATA_DIR = ROOT / "导入数据"
OUT_DIR = ROOT / "输出数据"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("项目根目录：", ROOT)
print("numpy", np.__version__, "| pandas", pd.__version__, "| scipy", sp_opt.__name__ and __import__("scipy").__version__)


项目根目录： D:\git项目
numpy 2.4.6 | pandas 3.1.0 | scipy 1.17.1


In [2]:
# =====================================================================
# 1. 配置区
# =====================================================================
CFG = dict(
    pop=120,          # 种群规模（GA）
    gen=400,          # 迭代代数（GA / SA）
    pc=0.8,           # 交叉概率
    pm=0.2,           # 变异概率
    seed=42,
    runs=5,           # 重复运行次数：用来报告结果分布，不是只报最好的一次
    tag="启发式算法",
    make_plots=True,
    save_excel=True,
)
print("配置：", CFG)


配置： {'pop': 120, 'gen': 400, 'pc': 0.8, 'pm': 0.2, 'seed': 42, 'runs': 5, 'tag': '启发式算法', 'make_plots': True, 'save_excel': True}


## 2. 目标函数与公共工具

这一节定义要优化的函数，以及多次运行的汇总方式。


In [3]:
# =====================================================================
# 2. 目标函数与公共工具
# =====================================================================
# 选一个多峰函数作为测试对象：有两个局部低谷，梯度法容易卡住。
def f_test(x):
    """(x0-3)^2 + (x1+1)^2 + 0.5*sin(3*x0)。

    这条函数有多个局部低谷，梯度法从不同初值会停在不同的谷底，
    正好用来检验启发式的全局搜索能力。
    用多起点精确求解得到的全局最优约 **-0.1969**，位置 x0≈3.45、x1=-1。
    """
    return (x[0] - 3) ** 2 + (x[1] + 1) ** 2 + 0.5 * np.sin(3 * x[0])


def clip(x, lo, hi):
    """把个体压回边界内：变异/扰动后必须做，否则搜索会跑出可行域。"""
    return np.minimum(np.maximum(x, lo), hi)


def summarize(results, name):
    """把多次运行的结果汇总：最好 / 最差 / 平均 / 标准差。"""
    vals = np.array([r["objective"] for r in results])
    return pd.DataFrame([{
        "算法": name, "运行次数": len(vals),
        "最好": round(vals.min(), 6), "最差": round(vals.max(), 6),
        "平均": round(vals.mean(), 6), "标准差": round(vals.std(ddof=1), 6) if len(vals) > 1 else 0,
        "平均用时(秒)": round(np.mean([r["elapsed"] for r in results]), 4),
    }])


## 3. 遗传算法


In [4]:
# =====================================================================
# 3. 遗传算法（实数编码）
# =====================================================================
def run_ga(fun, bounds, pop=100, gen=300, pc=0.8, pm=0.2, seed=0, maximize=False):
    """实数编码遗传算法。

    流程：初始化种群 → 算适应度 → 精英保留 → 锦标赛选择 → 算术交叉 → 高斯变异
    每一步都做了边界裁剪，保证解始终在 [lb, ub] 内。
    """
    t0 = time.time()
    rng = np.random.default_rng(seed)
    sign = -1.0 if maximize else 1.0
    F = lambda x: sign * float(fun(x))

    lo = np.array([b[0] for b in bounds], dtype=float)
    hi = np.array([b[1] for b in bounds], dtype=float)
    n = len(bounds)

    P = rng.uniform(lo, hi, size=(pop, n))
    fit = np.array([F(p) for p in P])
    history = [float(fit.min())]
    elite_n = max(2, pop // 10)                  # 精英保留，防止最优被交叉破坏

    for g in range(gen):
        order = np.argsort(fit)
        elite = P[order[:elite_n]]
        # 锦标赛选择：每次随机取两个，留适应度更好的
        parents = []
        for _ in range(pop - elite_n):
            i, j = rng.integers(0, pop, 2)
            parents.append(P[i] if fit[i] < fit[j] else P[j])
        parents = np.array(parents)

        # 算术交叉：两个父代按随机比例混合
        children = parents.copy()
        for k in range(0, len(children) - 1, 2):
            if rng.random() < pc:
                a = rng.random(n)
                children[k] = a * parents[k] + (1 - a) * parents[k + 1]
                children[k + 1] = a * parents[k + 1] + (1 - a) * parents[k]

        # 高斯变异：按 (hi-lo) 的比例扰动，再裁回边界
        mask = rng.random(children.shape) < pm
        children[mask] += rng.normal(0, 0.1, size=mask.sum()) * (hi - lo)[np.where(mask)[1]]
        children = clip(children, lo, hi)

        P = np.vstack([elite, children])
        fit = np.array([F(p) for p in P])
        history.append(float(fit.min()))

    best_idx = int(np.argmin(fit))
    return dict(x=P[best_idx], objective=sign * float(fit[best_idx]),
                history=history, elapsed=time.time() - t0,
                n_eval=(gen + 1) * pop)


ga_results = [run_ga(f_test, [(-5, 5), (-5, 5)], pop=CFG["pop"], gen=CFG["gen"],
                     pc=CFG["pc"], pm=CFG["pm"], seed=CFG["seed"] + i)
              for i in range(CFG["runs"])]
best_ga = min(ga_results, key=lambda r: r["objective"])
print("=== 遗传算法 ===")
print(summarize(ga_results, "GA").to_string(index=False))
print("\n最好解：x =", np.round(best_ga["x"], 4),
      "| 目标值 =", round(best_ga["objective"], 6),
      "| 评估次数 =", best_ga["n_eval"])
print("对照：多起点精确求解的全局最优 ≈ -0.1969，"
      f"GA 相差 {abs(best_ga['objective'] - (-0.196878)):.6f}")


=== 遗传算法 ===
算法  运行次数        最好        最差        平均  标准差  平均用时(秒)
GA     5 -0.196878 -0.196878 -0.196878  0.0   0.8866

最好解：x = [ 3.4504 -1.    ] | 目标值 = -0.196878 | 评估次数 = 48120
对照：多起点精确求解的全局最优 ≈ -0.1969，GA 相差 0.000000


## 4. 模拟退火


In [5]:
# =====================================================================
# 4. 模拟退火
# =====================================================================
def run_sa(fun, bounds, gen=2000, T0=1.0, T1=1e-3, seed=0, maximize=False):
    """模拟退火。

    核心是 Metropolis 准则：变差的解以 exp(-Δ/T) 的概率接受。
    温度 T 从 T0 指数降到 T1，前期敢跳出去、后期只做精细调整。
    """
    t0 = time.time()
    rng = np.random.default_rng(seed)
    sign = -1.0 if maximize else 1.0
    F = lambda x: sign * float(fun(x))

    lo = np.array([b[0] for b in bounds], dtype=float)
    hi = np.array([b[1] for b in bounds], dtype=float)
    n = len(bounds)

    cur = rng.uniform(lo, hi)
    cur_f = F(cur)
    best, best_f = cur.copy(), cur_f
    history = [float(best_f)]

    for g in range(gen):
        T = T0 * (T1 / T0) ** (g / max(gen - 1, 1))
        cand = clip(cur + rng.normal(0, 0.3, n) * (hi - lo), lo, hi)
        cand_f = F(cand)
        if cand_f < cur_f or rng.random() < np.exp(-(cand_f - cur_f) / max(T, 1e-12)):
            cur, cur_f = cand, cand_f
        if cur_f < best_f:
            best, best_f = cur.copy(), cur_f
        history.append(float(best_f))

    return dict(x=best, objective=sign * best_f, history=history,
                elapsed=time.time() - t0, n_eval=gen)


sa_results = [run_sa(f_test, [(-5, 5), (-5, 5)], gen=CFG["gen"] * 5,
                     seed=CFG["seed"] + i) for i in range(CFG["runs"])]
best_sa = min(sa_results, key=lambda r: r["objective"])
print("=== 模拟退火 ===")
print(summarize(sa_results, "SA").to_string(index=False))
print("\n最好解：x =", np.round(best_sa["x"], 4),
      "| 目标值 =", round(best_sa["objective"], 6))


=== 模拟退火 ===
算法  运行次数        最好        最差       平均      标准差  平均用时(秒)
SA     5 -0.193609 -0.173115 -0.18386 0.009372   0.0171

最好解：x = [ 3.4811 -0.9753] | 目标值 = -0.193609


## 5. 对比与收敛曲线


In [6]:
# =====================================================================
# 5. 与网格搜索对比 + 收敛曲线
# =====================================================================
# 网格搜索：把空间切细找一遍，作为"笨办法"的基准。
g = np.linspace(-5, 5, 401)
best_grid = (np.inf, None)
for x0 in g[::8]:                       # 步长 0.2，控制计算量
    for x1 in g[::8]:
        v = f_test(np.array([x0, x1]))
        if v < best_grid[0]:
            best_grid = (v, (x0, x1))
print(f"网格搜索（步长 0.2）：最优 {best_grid[0]:.4f} at {best_grid[1]}")
print(f"GA 最好 {best_ga['objective']:.4f} | SA 最好 {best_sa['objective']:.4f}")
print("\n注意：这里只是对照，不代表 GA 一定优于网格搜索；"
      "高维问题时网格会指数爆炸，启发式才有优势。")

fig = None
if CFG["make_plots"] and HAS_MPL:
    fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.8))
    for r in ga_results:
        axes[0].plot(r["history"], alpha=.55, lw=1)
    axes[0].set_xlabel("代数"); axes[0].set_ylabel("历史最优目标值")
    axes[0].set_title(f"GA 收敛曲线（{CFG['runs']} 次独立运行）")

    for r in sa_results:
        axes[1].plot(r["history"], alpha=.55, lw=1)
    axes[1].set_xlabel("迭代"); axes[1].set_ylabel("历史最优目标值")
    axes[1].set_title(f"SA 收敛曲线（{CFG['runs']} 次独立运行）")
    fig.tight_layout()

t_cmp = pd.concat([summarize(ga_results, "GA"), summarize(sa_results, "SA")],
                  ignore_index=True)
print("\n=== 两种算法对比 ===")
print(t_cmp.to_string(index=False))


网格搜索（步长 0.2）：最优 -0.1899 at (np.float64(3.4000000000000004), np.float64(-1.0))
GA 最好 -0.1969 | SA 最好 -0.1936

注意：这里只是对照，不代表 GA 一定优于网格搜索；高维问题时网格会指数爆炸，启发式才有优势。

=== 两种算法对比 ===
算法  运行次数        最好        最差        平均      标准差  平均用时(秒)
GA     5 -0.196878 -0.196878 -0.196878 0.000000   0.8866
SA     5 -0.193609 -0.173115 -0.183860 0.009372   0.0171


## 6. 结果导出


In [7]:
# =====================================================================
# 结果导出：CSV + 多 Sheet Excel + 图
# =====================================================================
def export_result(tables, cfg, fig=None, fig_name="结果图"):
    """把结果写成文件，返回写出的路径。

    tables : {Sheet 名: DataFrame}，第一张建议放"摘要"
    cfg    : 配置字典，用 tag / make_plots / save_excel 三个键
    fig    : 传了 matplotlib 图对象就保存；cfg["make_plots"]=False 时连图都不生成
    """
    d = OUT_DIR / f"优化_{cfg['tag']}"
    d.mkdir(parents=True, exist_ok=True)
    written = {}

    first = next(iter(tables.values()))
    p_csv = d / f"{cfg['tag']}_主表.csv"
    first.to_csv(p_csv, index=False, encoding="utf-8-sig")
    written["主表"] = p_csv

    if cfg.get("save_excel", True):
        p_xlsx = d / f"{cfg['tag']}_结果汇总.xlsx"
        with pd.ExcelWriter(p_xlsx, engine="openpyxl") as w:
            for name, frame in tables.items():
                frame.to_excel(w, sheet_name=str(name)[:28], index=False)
        written["汇总"] = p_xlsx
        print(f"[表格] {p_xlsx.name}（{len(tables)} 个 Sheet）")

    if fig is not None and cfg.get("make_plots", True) and HAS_MPL:
        p_png = d / f"{cfg['tag']}_{fig_name}.png"
        fig.savefig(p_png, dpi=150, bbox_inches="tight")
        if "agg" not in matplotlib.get_backend().lower():   # 无界面后端不弹窗，也不报警告
            plt.show()
        plt.close(fig)          # 关掉画布：反复运行不会在内存里堆图
        written["图片"] = p_png
        print(f"[图片] {p_png.name}")

    print("[输出] ", d)
    return written


In [8]:
# =====================================================================
# 6. 结果导出
# =====================================================================
written = export_result(
    {"算法对比": t_cmp,
     "GA最好解": pd.DataFrame({"变量": ["x1", "x2"], "取值": np.round(best_ga["x"], 4)}),
     "SA最好解": pd.DataFrame({"变量": ["x1", "x2"], "取值": np.round(best_sa["x"], 4)}),
     "GA多次运行": pd.DataFrame([{"运行": i + 1, "目标值": round(r["objective"], 6),
                              "用时(秒)": round(r["elapsed"], 4)}
                             for i, r in enumerate(ga_results)])},
    CFG, fig=fig, fig_name="收敛曲线")
print(written)


[表格] 启发式算法_结果汇总.xlsx（4 个 Sheet）
[图片] 启发式算法_收敛曲线.png
[输出]  D:\git项目\输出数据\优化_启发式算法
{'主表': WindowsPath('D:/git项目/输出数据/优化_启发式算法/启发式算法_主表.csv'), '汇总': WindowsPath('D:/git项目/输出数据/优化_启发式算法/启发式算法_结果汇总.xlsx'), '图片': WindowsPath('D:/git项目/输出数据/优化_启发式算法/启发式算法_收敛曲线.png')}


换题目时按三步走：

1. 数据放进 `导入数据/`（csv / xlsx 都行）；
2. 改本 Notebook 第 1 个代码单元里的 `CFG`：文件名、列名映射、方向、输出名；
3. `Run All`，结果写到 `输出数据/优化_{tag}/`，同名覆盖，不会越跑越多。

**两种输入方式怎么选**

- 数据还脏（有空行、重复、999、文本数字、列名带空格）→ 走「原始数据」示例，让 `clean_frame` 处理；
- 数据已经清洗/标准化完（比如上一节 `数据学习.ipynb` 输出的 `final_cleaned.csv`）→ 走「已处理数据」示例，直接给矩阵或宽表。

不要混：**同一次建模只用一种入口**，否则等于把同一份数据预处理了两遍。

**组合优化（排班、路径、装箱）怎么用**：把解编码成整数数组（如每个任务的开工顺序），目标函数里解码并算分，交叉/变异改成交换两段或翻转一段即可。
